In [2]:
import mkl
mkl.set_num_threads(23)
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt
from itertools import combinations

%load_ext autoreload
%autoreload 2
%matplotlib widget
import matplotlib.pyplot as plt

plt.rcParams.update({'font.size': 8})
plt.rcParams.update({'pdf.fonttype':42})
plt.rcParams.update({'figure.max_open_warning': 0})
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.spines.top'] = False


better_result=dict(x=np.load('/data1/projects/dumoulinlab/Lab_members/Marco/NFsim/de_fitting_hcpmeg_subc_dti50_4.npy'))

Laplacian_eigenvalues = np.load('/data1/projects/dumoulinlab/Lab_members/Marco/NFsim/eigvals_DTI_fgCCfix_subcortex_dti50.npy')



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [29]:




def GraphKernel(x,t,type='Gaussian', a=10**3, b=10, c=0, prime=False):
    if t<0:
        #print("Need positive kernel parameter")
        return
    else:
        if type=='Gaussian':
            #add prefactor to make kernel of unitary height
            return np.exp(t*x)#*2*np.sqrt(t*np.pi)
        elif type=='Exponential':
            return t/(t-x) #*2*t
        elif type=='Pyramid':
            #return t*(np.sinc(t*np.sqrt(-x)/(2*np.pi)))**2
            return np.sinc(np.sqrt(-x)/(2*np.pi*t))**2
        elif type=='Rectangle':
            #Note: significant Gibbs effect makes this not advisable
            rect=np.sinc(np.sqrt(-x)/(2*np.pi*t))
            #rect[-990:]=0
            return rect
        elif type=='Mexican Hat':
            return -x*np.exp(t*x) #*2*np.sqrt(t*np.pi)  #*2*t
        elif type=='Damped Wave':
            delta_dw = b**2 + 4*a*(x-c)
            if a>0:# and np.all((delta_dw >=0)):

                r_1=(-b+np.emath.sqrt(delta_dw))/(2*a)
                r_2=(-b-np.emath.sqrt(delta_dw))/(2*a)
                rdiff = r_1-r_2
                #print(rdiff.imag)
                if np.all(rdiff != 0):


                    Damped_Wave_Kernel=(r_1*np.exp(r_2*t)-r_2*np.exp(r_1*t))/(rdiff)

                    if prime:
                        Damped_Wave_Kernel_prime=(np.exp(r_1*t)-np.exp(r_2*t))/(rdiff)

                        return Damped_Wave_Kernel.real.astype('float64'), Damped_Wave_Kernel_prime.real.astype('float64')
                    else:
                        return Damped_Wave_Kernel.real.astype('float64')
                else:
                    print('lol')
                    return np.nan
            else:
                print('lolaaa')
                return np.nan

def H_Simple_Steady_State(alpha_EE=1, alpha_IE=1, alpha_EI=1, alpha_II=1, d_e=1, d_i=1, P=0, Q=0):
    #generate multiple initial conditions to find all steady states
    initial_guesses = 20
    ##print("%.3g %.3g %.3g %.3g %.3g %.3g %.3g %.3g"%(alpha_EE, alpha_IE, alpha_EI, alpha_II, d_e, d_i, P, Q))

    # x0 = np.zeros((2,initial_guesses))
    # x0[:,1] = np.array([1/(2*d_e), 1/(2*d_i)])
    # x0[:,2] = np.random.rand(2)
    # x0[:,3] = np.random.rand(2)
    # x0[:,4] = np.random.rand(2)

    x0 = np.random.rand(2,initial_guesses)
    results = []

    success = False

    def f(x, alpha_EE, alpha_IE, alpha_EI, alpha_II, d_e, d_i, P, Q):
        d = np.array([[d_e,0],[0,d_i]], dtype=float)
        alpha = np.array([[alpha_EE,-alpha_IE],[alpha_EI,-alpha_II]], dtype=float)
        X = np.array([P,Q], dtype=float)

        SS_EQ = - np.dot(d,x) + sp.special.expit(np.dot(alpha,x) + X)
        return SS_EQ

    for i in range(initial_guesses):
        steady_state_res = sp.optimize.root(f,x0[:,i],args=(alpha_EE,alpha_IE,alpha_EI,alpha_II,d_e,d_i,P,Q),
                                          method='lm',
                                          options={'ftol':1e-12})
        #print(steady_state_res['x'])
        #print(steady_state_res['fun'][0]-steady_state_res['fun'][1])
        fun = steady_state_res['fun'][0]-steady_state_res['fun'][1]
        steady_state = steady_state_res['x']


        if np.all(steady_state>0.001) and np.all(steady_state<0.999) and np.abs(fun) <1e-9: # and np.linalg.norm(steady_state[1]['fvec'],ord=1)<=1e-20:
            results.append(steady_state)
            success=True


    #select and importantly sort the unique, acceptable results
    if success==True:
        results = np.array(results)
        #print(np.unique(results.round(5), axis=0, return_index=True))
        #results = results[:,~np.all(np.isnan(results), axis=0)]

        finals = results[np.unique(results.round(5), axis=0, return_index=True)[1]]

        return finals.T, success
    else:
  #      #print("No positive, exact solutions were found")
        return None, success


# -----------------------
# Placeholder for Jacobian computation
# -----------------------
def compute_jacobian(params, ss):
    """
    Compute the Jacobian matrix for the Wilson-Cowan model at the fixed point.
    Parameters:
        params (dict): Dictionary of model parameters.
    Returns:
        J (np.ndarray): Jacobian matrix (40,000 x 40,000 for 20,000 nodes with E/I).
    """
    Graph_Kernel = 'Damped Wave'

    aEE = params['aEE']
    aIE = params['aIE']
    aEI = params['aEI']
    aII = params['aII']
    dE = params['dE']
    dI = params['dI']
    P = params['P']
    Q = params['Q']
    sEE = params['sEE']
    sIE = params['sIE']
    sEI = params['sEI']
    sII = params['sII']
    tE = params['tE']
    tI = params['tI']
    aDWEE = params['aDWEE']
    aDWIE = params['aDWIE']
    aDWEI = params['aDWEI']
    aDWII = params['aDWII']
    bDWEE = params['bDWEE']
    bDWIE = params['bDWIE']
    bDWEI = params['bDWEI']
    bDWII = params['bDWII']

    t_EE = (0.5*sEE**2)
    t_IE = (0.5*sIE**2)
    t_EI = (0.5*sEI**2)
    t_II = (0.5*sII**2)

    eigs=Laplacian_eigenvalues

    Jacobian_eigenvalues=np.zeros((len(eigs),2),dtype=complex)

    Ess = ss[0]
    Iss = ss[1]

    ass = dE*Ess*(1-dE*Ess)
    bss = dI*Iss*(1-dI*Iss)

    K_EE = GraphKernel(eigs,t_EE,type=Graph_Kernel,a=aDWEE,b=bDWEE)
    K_IE = GraphKernel(eigs,t_IE,type=Graph_Kernel,a=aDWIE,b=bDWIE)
    K_EI = GraphKernel(eigs,t_EI,type=Graph_Kernel,a=aDWEI,b=bDWEI)
    K_II = GraphKernel(eigs,t_II,type=Graph_Kernel,a=aDWII,b=bDWII)

    return np.array([[-dE/tE + aEE*ass*K_EE/tE, -aIE*ass*K_IE/tE],[aEI*bss*K_EI/tI,-dI/tI -aII*bss*K_II/tI]]).T

def leading_eigenvalue(jacobian):
    """
    Compute the leading eigenvalue (largest real part) of the Jacobian.
    """
    eigvals_real = np.ravel(np.linalg.eigvals(jacobian)).real
    return np.max(eigvals_real)


# -----------------------
# Sensitivity Analysis to Identify Relevant Parameters
# -----------------------
def sensitivity_analysis(params, perturbation_scales=[0.01, 0.02], max_attempts=3, threshold=0.1):
    """
    Identify parameters that significantly affect stability using a staircase approach.
    Parameters:
        params (dict): Model parameters.
        perturbation_scales (list): Initial perturbation scales (e.g., 1%, 2%).
        max_attempts (int): Max number of refinement steps per parameter.
        threshold (float): Change in Re(λ_max) to consider parameter sensitive.
    Returns:
        sensitive_params (list): List of parameter names that affect stability.
    """
    sensitive_params = []
    # Compute steady states for base parameters
    steady_states, success = H_Simple_Steady_State(
        alpha_EE=params['aEE'], alpha_IE=params['aIE'], alpha_EI=params['aEI'], alpha_II=params['aII'],
        d_e=params['dE'], d_i=params['dI'], P=params['P'], Q=params['Q']
    )
    if not success or steady_states.shape[1] == 0:
        print("No valid steady states found for base parameters.")
        return sensitive_params

    # Select steady state zero as the reference
    reference_ss = steady_states[:, 0]
    print(reference_ss)
    J = compute_jacobian(params, reference_ss)
    base_eig = leading_eigenvalue(J).real

    for param in params:
        significant = False
        current_value = params[param]
        scales = perturbation_scales.copy()
        previous_ss = reference_ss

        for attempt in range(max_attempts):
            for scale in scales:
                for direction in [1, -1]:
                    test_params = params.copy()
                    test_params[param] = current_value * (1 + direction * scale)
                    # Recompute steady states for perturbed parameters
                    steady_states, success = H_Simple_Steady_State(
                        alpha_EE=test_params['aEE'], alpha_IE=test_params['aIE'],
                        alpha_EI=test_params['aEI'], alpha_II=test_params['aII'],
                        d_e=test_params['dE'], d_i=test_params['dI'],
                        P=test_params['P'], Q=test_params['Q']
                    )
                    if not success or steady_states is None or steady_states.shape[1] == 0:
                        continue

                    # Find the steady state closest to the previous one
                    distances = [np.linalg.norm(ss - previous_ss) for ss in steady_states.T]
                    closest_ss = steady_states.T[np.argmin(distances)]
                    J = compute_jacobian(test_params, closest_ss)
                    eig = leading_eigenvalue(J).real
                    delta = abs(eig - base_eig)
                    if delta > threshold or (base_eig * eig < 0):  # Significant change or stability flip
                        significant = True
                        previous_ss = closest_ss  # Update previous steady state
                        break
                if significant:
                    break
            if significant:
                sensitive_params.append(param)
                break
            scales = [s / 2 for s in scales]

    return sensitive_params

def linear_stability_analysis(params, max_attempts=3, num_points_1d=51, num_points_2d=51, scale_range=None, output_dir="stability_plots"):
    """
    Perform a principled linear stability analysis, tracking all steady states.
    Parameters:
        params (dict): Model parameters.
        max_attempts (int): Max attempts for sensitivity analysis.
        num_points_1d (int): Points for 1D scans.
        num_points_2d (int): Points per axis for 2D scans.
        output_dir (str): Directory to save plots.
    """
    import os
    os.makedirs(output_dir, exist_ok=True)

    # Step 1: Sensitivity analysis to find relevant parameters
    print("Performing sensitivity analysis...")
    sensitive_params = sensitivity_analysis(params, max_attempts=max_attempts)
    print(f"Sensitive parameters: {sensitive_params}")

    if not sensitive_params:
        print("No sensitive parameters found. Try increasing perturbation scales or threshold.")
        return

    # Compute initial steady states for reference
    steady_states, success = H_Simple_Steady_State(
        alpha_EE=params['aEE'], alpha_IE=params['aIE'], alpha_EI=params['aEI'], alpha_II=params['aII'],
        d_e=params['dE'], d_i=params['dI'], P=params['P'], Q=params['Q']
    )
    if not success or steady_states is None or steady_states.shape[1] == 0:
        print("No valid steady states found for base parameters.")
        return
    
    all_thresholds = []

    instability_thresholds = []

    # Step 2: 1D Stability Scans for each steady state
    print("Generating 1D stability scans...")
    for param in sensitive_params:  # Limit to top 3 for practicality
        for ss_idx in range(1):#steady_states.shape[1]):
            reference_ss = steady_states[:, ss_idx]
            values, lead_reals = stability_scan_1d(params, param, num_points=num_points_1d, reference_ss=reference_ss, scale_range=scale_range)
            plt.figure(figsize=(6, 6))
            plt.plot(values, lead_reals, label=f"Re(λ_max) Steady State {ss_idx}", c='k', ls='-',lw=2, marker='s',mfc='grey',mec='k',markersize=4)
            plt.axhline(0, color="k", linestyle="--", label="Stability boundary", lw=0.75, alpha=0.75)
            # Set 5 ticks as percentages
            ticks = np.linspace(params[param] * (1 + scale_range[param][0]),
                                params[param] * (1 + scale_range[param][1]), 5)
            plt.xticks(ticks, [f"{(t / params[param] - 1) * 100:.2f}%" for t in ticks])
            plt.xlabel(param)
            plt.ylabel("Re(λ_max)")
            plt.ylim(-5,10)
            plt.plot(params[param],lead_reals[np.argmin(np.abs(params[param]-values))],marker='*',mfc='orange',mec='k',markersize=16)
            plt.title(f"1D Stability Scan: {param} (Steady State {reference_ss})")
            plt.legend()
            plt.savefig(f"{output_dir}/1d_scan_{param}_ss{ss_idx}.pdf", dpi=600, transparent=True) 
            plt.close()

            valid = ~np.isnan(lead_reals)
            if not np.any(valid):
                continue
            values_valid = values[valid]
            lead_reals_valid = lead_reals[valid]
            base_value = params[param]
            base_idx = np.argmin(np.abs(values_valid - base_value))
            base_eig = lead_reals_valid[base_idx]
            if base_eig >= 0:
                print(f"Base state is unstable for {param}")
                continue
            #cross_indices = np.where(np.diff(np.sign(lead_reals_valid)) > 0)[0]
            cross_indices = np.where(np.sign(lead_reals_valid[:-1]) * np.sign(lead_reals_valid[1:]) < 0)[0]
            for ci in cross_indices:
                v1 = values_valid[ci]
                v2 = values_valid[ci+1]
                e1 = lead_reals_valid[ci]
                e2 = lead_reals_valid[ci+1]
                zero_v = v1 - e1 * (v2 - v1) / (e2 - e1)
                perc = (zero_v - base_value) / base_value * 100
                all_thresholds.append((param, perc))
                if abs(perc) < 1:
                    instability_thresholds.append((param, perc))

    unique_params = set([p for p, _ in instability_thresholds])
    N = len(unique_params)
    if N > 0:
        specific = ", ".join([f"{'+' if perc > 0 else ''}{perc:.2f}% in {param}" for param, perc in sorted(instability_thresholds, key=lambda x: x[0])])
        print(f"there are {N} model parameters whereby a change smaller than 1% is sufficient to cause instability ({', '.join(sorted(unique_params))}). specifically ({specific})")
    else:
        print("No parameters found where <1% change causes instability.")

    other_thresholds = [t for t in all_thresholds if abs(t[1]) >= 1]
    unique_other = set([p for p, _ in other_thresholds])
    M = len(unique_other)
    if M > 0:
        specific_other = ", ".join([f"{'+' if perc > 0 else ''}{perc:.2f}% in {param}" for param, perc in sorted(other_thresholds, key=lambda x: x[0])])
        print(f"there are {M} model parameters whereby a change larger than or equal to 1% is sufficient to cause instability ({', '.join(sorted(unique_other))}). specifically ({specific_other})")
    else:
        print("No additional parameters found where >=1% change causes instability.")

    # Step 3: 2D Stability Scans for pairs of sensitive parameters
    if len(sensitive_params) >= 2:
        print("Generating 2D stability scans...")
        param_pairs = list(combinations(sensitive_params[:4], 2))  # Limit to top 3-4 parameters[4:-1]
        for p1, p2 in param_pairs:
            for ss_idx in range(1):#steady_states.shape[1]):
                reference_ss = steady_states[:, ss_idx]
                p1_vals, p2_vals, stab_matrix = stability_scan_2d(params, p1, p2, num_points=num_points_2d, reference_ss=reference_ss, scale_range=scale_range)
                plt.figure(figsize=(7, 6))
                cmap = plt.cm.get_cmap('turbo')  # Copy the RdBu_r colormap
                cmap.set_bad(color='black')  # Set NaN values to black
                plt.pcolormesh(p1_vals, p2_vals, stab_matrix.T, vmin=-3, vmax=3, cmap=cmap)
                plt.plot(params[p1],params[p2],marker='*',mfc='orange',mec='k',markersize=20)
                plt.colorbar(label="Re(λ_max)")
                plt.xlabel(p1)
                plt.ylabel(p2)
                # Set 5 ticks as percentages for x and y axes
                p1_ticks = np.linspace(params[p1] * (1 + scale_range[p1][0]),
                                      params[p1] * (1 + scale_range[p1][1]), 5)
                p2_ticks = np.linspace(params[p2] * (1 + scale_range[p2][0]),
                                      params[p2] * (1 + scale_range[p2][1]), 5)
                plt.xticks(p1_ticks, [f"{(t / params[p1] - 1) * 100:.2f}%" for t in p1_ticks])
                plt.yticks(p2_ticks, [f"{(t / params[p2] - 1) * 100:.2f}%" for t in p2_ticks])
                plt.title(f"2D Stability Phase Portrait: {p1} vs {p2} (Steady State {reference_ss})")
                plt.savefig(f"{output_dir}/2d_scan_{p1}_vs_{p2}_ss{ss_idx}.pdf", dpi=600, transparent=True) 
                plt.close()



    print(f"Plots saved in {output_dir}")

# -----------------------
# 1D Parameter Sweep (Adaptive)
# -----------------------
def stability_scan_1d(params, param_name, num_points=51, scale_range=None, reference_ss=None):
    """
    Perform a 1D stability scan for a single parameter, tracking a specific steady state.
    Parameters:
        params (dict): Model parameters.
        param_name (str): Parameter to vary.
        num_points (int): Number of points in the scan.
        scale_range (tuple): Relative range for parameter variation (e.g., ±10%).
        reference_ss (np.ndarray): Reference steady state to track.
    Returns:
        values (np.ndarray): Parameter values scanned.
        leading_real (np.ndarray): Real parts of leading eigenvalues for tracked steady state.
    """
    base_value = params[param_name]

    if scale_range is None:
      scale_range = {param_name:(-0.01,0.01)}
    else:
      if param_name not in scale_range:
        scale_range[param_name] = (-0.01,0.01)

    values = np.linspace(base_value * (1 + scale_range[param_name][0]), base_value * (1 + scale_range[param_name][1]), num_points)
    leading_real = []
    previous_ss = reference_ss

    for v in values:
        test_params = params.copy()
        test_params[param_name] = v
        # Recompute steady states
        steady_states, success = H_Simple_Steady_State(
            alpha_EE=test_params['aEE'], alpha_IE=test_params['aIE'],
            alpha_EI=test_params['aEI'], alpha_II=test_params['aII'],
            d_e=test_params['dE'], d_i=test_params['dI'],
            P=test_params['P'], Q=test_params['Q']
        )
        if not success or steady_states is None or steady_states.shape[1] == 0:
            leading_real.append(np.nan)
            continue

        # Find the steady state closest to the previous one
        distances = [np.linalg.norm(ss - previous_ss) for ss in steady_states.T]
        closest_ss = steady_states.T[np.argmin(distances)]

        if np.min(distances)>0.05:
          print(f"too far state: {closest_ss}; dist ({np.min(distances)})")
          leading_real.append(np.nan)
          continue

        J = compute_jacobian(test_params, closest_ss)
        eig = leading_eigenvalue(J).real
        leading_real.append(eig)
        #previous_ss = closest_ss  # Update previous steady state

    return values, np.array(leading_real)

# -----------------------
# 2D Parameter Scan
# -----------------------
def stability_scan_2d(params, p1, p2, num_points=51, scale_range=None, reference_ss=None):
    """
    Perform a 2D stability scan for two parameters, tracking a specific steady state.
    Parameters:
        params (dict): Model parameters.
        p1, p2 (str): Names of parameters to vary.
        num_points (int): Number of points per axis.
        scale_range (tuple): Relative range for parameter variation.
        reference_ss (np.ndarray): Reference steady state to track.
    Returns:
        p1_vals, p2_vals (np.ndarray): Parameter values scanned.
        stability (np.ndarray): Real parts of leading eigenvalues for tracked steady state.
    """
    p1_base = params[p1]
    p2_base = params[p2]

    if scale_range is None:
      scale_range = {p:(-0.01,0.01) for p in [p1,p2]}
    else:
      if p1 not in scale_range:
        scale_range[p1] = (-0.01,0.01)
      if p2 not in scale_range:
        scale_range[p2] = (-0.01,0.01)


    p1_vals = np.linspace(p1_base * (1 + scale_range[p1][0]), p1_base * (1 + scale_range[p1][1]), num_points)
    p2_vals = np.linspace(p2_base * (1 + scale_range[p2][0]), p2_base * (1 + scale_range[p2][1]), num_points)
    stability = np.zeros((num_points, num_points))
    previous_ss = reference_ss

    print(f"2d scan {reference_ss}: {p1} {p2}")

    for i, v1 in enumerate(p1_vals):
        for j, v2 in enumerate(p2_vals):
            test_params = params.copy()
            test_params[p1] = v1
            test_params[p2] = v2
            # Recompute steady states
            steady_states, success = H_Simple_Steady_State(
                alpha_EE=test_params['aEE'], alpha_IE=test_params['aIE'],
                alpha_EI=test_params['aEI'], alpha_II=test_params['aII'],
                d_e=test_params['dE'], d_i=test_params['dI'],
                P=test_params['P'], Q=test_params['Q']
            )
            if not success or steady_states is None or steady_states.shape[1] == 0:
                stability[i, j] = np.nan
                continue

            # Find the steady state closest to the previous one
            distances = [np.linalg.norm(ss - previous_ss) for ss in steady_states.T]
            closest_ss = steady_states.T[np.argmin(distances)]

            if np.min(distances)>0.05:
              print(f"too far state: {closest_ss}; dist ({np.min(distances)})")
              stability[i, j] = np.nan
              continue

            J = compute_jacobian(test_params, closest_ss)
            eig = leading_eigenvalue(J).real
            stability[i, j] = eig
            #previous_ss = closest_ss  # Update previous steady state

    return p1_vals, p2_vals, stability

# -----------------------
# Example Usage
# -----------------------
if __name__ == "__main__":
    # User-specified base parameters (replace with your actual parameters)
    base_params= dict()
    scale_range = dict()

    base_params['aEE'] = better_result['x'][0]
    base_params['aIE'] = better_result['x'][1]
    base_params['aEI'] = better_result['x'][2]
    base_params['aII'] = better_result['x'][3]
    base_params['dE'] = better_result['x'][4]
    base_params['dI'] = better_result['x'][5]
    base_params['P'] = better_result['x'][6]
    base_params['Q'] = better_result['x'][7]
    base_params['sEE'] = better_result['x'][8]
    base_params['sIE'] = better_result['x'][9]
    base_params['sEI'] = better_result['x'][10]
    base_params['sII'] = better_result['x'][11]
    base_params['tE'] = better_result['x'][12]
    base_params['tI'] = better_result['x'][13]
    base_params['aDWEE'] = better_result['x'][14]
    base_params['aDWIE'] = better_result['x'][15]
    base_params['aDWEI'] = better_result['x'][16]
    base_params['aDWII'] = better_result['x'][17]
    base_params['bDWEE'] = better_result['x'][18]
    base_params['bDWIE'] = better_result['x'][19]
    base_params['bDWEI'] = better_result['x'][20]
    base_params['bDWII'] = better_result['x'][21]

    # Run sensitivity analysis first to determine perturbation scales
    # print("Performing preliminary sensitivity analysis to set scale ranges...")
    # sensitive_params = sensitivity_analysis(base_params, perturbation_scales=[0.01, 0.02], max_attempts=3, threshold=0.1)
    
    # # Compute steady states for base parameters
    # steady_states, success = H_Simple_Steady_State(
    #     alpha_EE=base_params['aEE'], alpha_IE=base_params['aIE'], alpha_EI=base_params['aEI'], alpha_II=base_params['aII'],
    #     d_e=base_params['dE'], d_i=base_params['dI'], P=base_params['P'], Q=base_params['Q']
    # )
    # if not success or steady_states is None or steady_states.shape[1] == 0:
    #     print("No valid steady states found for base parameters.")
    #     exit()

    # reference_ss = steady_states[:, 0]

    # # Set scale_range based on distance to instability
    # for param in sensitive_params:
    #     if param not in scale_range:
    #         values, lead_reals = stability_scan_1d(base_params, param, num_points=41, scale_range={param: (-0.05, 0.05)}, reference_ss=reference_ss)
    #         valid = ~np.isnan(lead_reals)
    #         if not np.any(valid):
    #             print(f"No valid eigenvalues for {param}, using default range")
    #             scale_range[param] = (-0.05, 0.05)
    #             continue
    #         values_valid = values[valid]
    #         lead_reals_valid = lead_reals[valid]
    #         base_value = base_params[param]
    #         base_idx = np.argmin(np.abs(values_valid - base_value))
    #         base_eig = lead_reals_valid[base_idx]
    #         if base_eig >= 0:
    #             print(f"Base state is unstable for {param}, using default range")
    #             scale_range[param] = (-0.05, 0.05)
    #             continue
    #         cross_indices = np.where(np.sign(lead_reals_valid[:-1]) * np.sign(lead_reals_valid[1:]) < 0)[0]
    #         if len(cross_indices) == 0:
    #             print(f"No zero crossing for {param}, using default range")
    #             scale_range[param] = (-0.05, 0.05)
    #             continue
    #         min_perc = float('inf')
    #         for ci in cross_indices:
    #             v1 = values_valid[ci]
    #             v2 = values_valid[ci+1]
    #             e1 = lead_reals_valid[ci]
    #             e2 = lead_reals_valid[ci+1]
    #             zero_v = v1 - e1 * (v2 - v1) / (e2 - e1)
    #             perc = abs((zero_v - base_value) / base_value)
    #             if perc < min_perc:
    #                 min_perc = perc
    #         scale_range[param] = (-10 * min_perc, 10 * min_perc)


    # Override specific parameters with user-defined ranges
    # scale_range['aEE'] = (-0.01*2, 0.01*2)
    # scale_range['aEI'] = (-0.02*2, 0.02*2)
    # scale_range['aIE'] = (-0.001*2, 0.001*2)
    # scale_range['aII'] = (-0.001*2, 0.001*2)

    # Run the analysis
    linear_stability_analysis(base_params, max_attempts=3, num_points_1d=61, num_points_2d=61, scale_range=scale_range,
                              output_dir='stability_plots_turbo33_FIXED1_')

Performing sensitivity analysis...
[0.00532051 0.08362476]
Sensitive parameters: ['aEE', 'aIE', 'aEI', 'aII', 'dE', 'dI', 'P', 'Q', 'sEE']
Generating 1D stability scans...
there are 7 model parameters whereby a change smaller than 1% is sufficient to cause instability (P, Q, aEE, aIE, aII, dE, dI). specifically (+0.06% in P, -0.15% in Q, +0.44% in aEE, -0.05% in aIE, +0.05% in aII, -0.69% in dE, +0.36% in dI)
No additional parameters found where >=1% change causes instability.
Generating 2D stability scans...
2d scan [0.00532051 0.08362476]: aEE aIE


/tmp/ipykernel_287320/483199165.py:343: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.
  cmap = plt.cm.get_cmap('turbo')  # Copy the RdBu_r colormap


2d scan [0.00532051 0.08362476]: aEE aEI
2d scan [0.00532051 0.08362476]: aEE aII
2d scan [0.00532051 0.08362476]: aIE aEI
2d scan [0.00532051 0.08362476]: aIE aII
2d scan [0.00532051 0.08362476]: aEI aII
Plots saved in stability_plots_turbo33_FIXED1_
